In [ ]:
import os
import h5py
import time
import numpy
import pandas
from rail import core
from matplotlib import pyplot
from matplotlib import colors
from photerr import LsstErrorModel
from matplotlib.gridspec import GridSpec
from rail.estimation.algos import somoclu_som

In [ ]:
index = 0
tag = 'Y1'
folder="/global/cfs/cdirs/lsst/groups/MCP/CosmoCloud/ZCloud/"

Application datasets

In [ ]:
# Start
start = time.time()

# Path
dataset_folder = os.path.join(folder, 'DATASET/')
os.makedirs(os.path.join(dataset_folder, '{}'.format(tag)), exist_ok=True)
os.makedirs(os.path.join(dataset_folder, '{}/APPLICATION/'.format(tag)), exist_ok=True)

# Load
with h5py.File(os.path.join(dataset_folder, '{}/OBSERVATION/OBSERVATION.hdf5'.format(tag)), 'r') as file:
    observation_dataset = {key: file[key][...] for key in file.keys()}

# Error
error_model = LsstErrorModel(
    absFlux=True,
    ndMode='sigLim', 
    majorCol='major', 
    minorCol='minor', 
    decorrelate=True,
    extendedType='auto',
    nYrObs=int(tag[1:]), 
    renameDict={
        'u': 'mag_u_lsst',
        'g': 'mag_g_lsst',
        'r': 'mag_r_lsst',
        'i': 'mag_i_lsst',
        'z': 'mag_z_lsst',
        'y': 'mag_y_lsst'
    }
)

# Application
application_dataset = dict(error_model(pandas.DataFrame(observation_dataset), random_state=index))

# SOM
data_store = core.stage.RailStage.data_store
data_store.__class__.allow_overwrite = True

model_name = os.path.join(dataset_folder, '{}/SOM/INFORM.pkl'.format(tag))
model = data_store.read_file(key='model', path=model_name, handle_class=core.data.ModelHandle)()

chunk = 100000
application_size = len(application_dataset['redshift'])
application_cell_coordinate = numpy.zeros((application_size, 2), dtype=numpy.int32)

for m in range(application_size // chunk + 1):
    begin = m * chunk
    stop = min((m + 1) * chunk, application_size)
    application = {key: application_dataset[key][begin: stop].astype(numpy.float32) for key in model['usecols']}
    
    application_column = somoclu_som._computemagcolordata(data=application, mag_column_name=model['ref_column'], column_names=model['usecols'], colusage=model['column_usage'])
    application_cell_coordinate[begin: stop, :] = somoclu_som.get_bmus(model['som'], application_column)

application_cell_coordinate1 = application_cell_coordinate[:, 0]
application_cell_coordinate2 = application_cell_coordinate[:, 1]
application_cell_id = numpy.ravel_multi_index(numpy.transpose(application_cell_coordinate), (model['n_rows'], model['n_columns']))

cell_size = model['n_rows'] * model['n_columns']
application_cell_count = numpy.bincount(application_cell_id, minlength=cell_size)
application_cell_z_true = numpy.divide(numpy.bincount(application_cell_id, weights=application_dataset['redshift_true'], minlength=cell_size), application_cell_count, out=numpy.ones(cell_size) * numpy.nan, where=application_cell_count != 0)

# Save
with h5py.File(os.path.join(dataset_folder, '{}/APPLICATION/DATA{}.hdf5'.format(tag, index)), 'w') as file:
    file.create_group('meta')
    file['meta'].create_dataset('cell_size', data=cell_size, dtype=numpy.int32)
    file['meta'].create_dataset('cell_size1', data=model['n_rows'], dtype=numpy.int32)
    file['meta'].create_dataset('cell_size2', data=model['n_columns'], dtype=numpy.int32)
    
    file['meta'].create_dataset('cell_id', data=application_cell_id, dtype=numpy.int32)
    file['meta'].create_dataset('cell_coordinate1', data=application_cell_coordinate1, dtype=numpy.int32)
    file['meta'].create_dataset('cell_coordinate2', data=application_cell_coordinate2, dtype=numpy.int32)
    
    file['meta'].create_dataset('cell_count', data=application_cell_count, dtype=numpy.int32)
    file['meta'].create_dataset('cell_z_true', data=application_cell_z_true, dtype=numpy.float32)
    
    file.create_group('photometry')
    file['photometry'].create_dataset('redshift', data=application_dataset['redshift'], dtype=numpy.float32)
    file['photometry'].create_dataset('redshift_true', data=application_dataset['redshift_true'], dtype=numpy.float32)
    
    file['photometry'].create_dataset('mag_u_lsst', data=application_dataset['mag_u_lsst'], dtype=numpy.float32)
    file['photometry'].create_dataset('mag_g_lsst', data=application_dataset['mag_g_lsst'], dtype=numpy.float32)
    file['photometry'].create_dataset('mag_r_lsst', data=application_dataset['mag_r_lsst'], dtype=numpy.float32)
    file['photometry'].create_dataset('mag_i_lsst', data=application_dataset['mag_i_lsst'], dtype=numpy.float32)
    file['photometry'].create_dataset('mag_z_lsst', data=application_dataset['mag_z_lsst'], dtype=numpy.float32)
    file['photometry'].create_dataset('mag_y_lsst', data=application_dataset['mag_y_lsst'], dtype=numpy.float32)
    
    file['photometry'].create_dataset('mag_u_lsst_err', data=application_dataset['mag_u_lsst_err'], dtype=numpy.float32)
    file['photometry'].create_dataset('mag_g_lsst_err', data=application_dataset['mag_g_lsst_err'], dtype=numpy.float32)
    file['photometry'].create_dataset('mag_r_lsst_err', data=application_dataset['mag_r_lsst_err'], dtype=numpy.float32)
    file['photometry'].create_dataset('mag_i_lsst_err', data=application_dataset['mag_i_lsst_err'], dtype=numpy.float32)
    file['photometry'].create_dataset('mag_z_lsst_err', data=application_dataset['mag_z_lsst_err'], dtype=numpy.float32)
    file['photometry'].create_dataset('mag_y_lsst_err', data=application_dataset['mag_y_lsst_err'], dtype=numpy.float32)
    
    file.create_group('morphology')
    file['morphology'].create_dataset('ra', data=application_dataset['ra'], dtype=numpy.float32)
    file['morphology'].create_dataset('dec', data=application_dataset['dec'], dtype=numpy.float32)
    
    file['morphology'].create_dataset('id', data=application_dataset['id'], dtype=numpy.int32)
    file['morphology'].create_dataset('value', data=application_dataset['value'], dtype=numpy.int32)
    
    file['morphology'].create_dataset('mu', data=application_dataset['mu'], dtype=numpy.float32)
    file['morphology'].create_dataset('eta', data=application_dataset['eta'], dtype=numpy.float32)
    file['morphology'].create_dataset('sigma', data=application_dataset['sigma'], dtype=numpy.float32)
    
    file['morphology'].create_dataset('major', data=application_dataset['major'], dtype=numpy.float32)
    file['morphology'].create_dataset('minor', data=application_dataset['minor'], dtype=numpy.float32)
    
    file['morphology'].create_dataset('major_disk', data=application_dataset['major_disk'], dtype=numpy.float32)
    file['morphology'].create_dataset('major_bulge', data=application_dataset['major_bulge'], dtype=numpy.float32)
    
    file['morphology'].create_dataset('ellipticity_disk', data=application_dataset['ellipticity_disk'], dtype=numpy.float32)
    file['morphology'].create_dataset('ellipticity_bulge', data=application_dataset['ellipticity_bulge'], dtype=numpy.float32)
    file['morphology'].create_dataset('bulge_to_total_ratio', data=application_dataset['bulge_to_total_ratio'], dtype=numpy.float32)

# Duration
end = time.time()
duration = (end - start) / 60
print('Time: {:.2f} minutes'.format(duration))

Degradation datasets

In [ ]:
# Start
start = time.time()

# Path
dataset_folder = os.path.join(folder, 'DATASET/')
os.makedirs(os.path.join(dataset_folder, '{}'.format(tag)), exist_ok=True)
os.makedirs(os.path.join(dataset_folder, '{}/DEGRADATION/'.format(tag)), exist_ok=True)

# Application
application_dataset = {
    'meta': {},
    'morphology': {},
    'photometry': {}
}

with h5py.File(os.path.join(dataset_folder, '{}/APPLICATION/DATA{}.hdf5'.format(tag, index)), 'r') as file:
    application_dataset['meta'] = {key: file['meta'][key][...] for key in file['meta'].keys()}
    application_dataset['morphology'] = {key: file['morphology'][key][...] for key in file['morphology'].keys()}
    application_dataset['photometry'] = {key: file['photometry'][key][...] for key in file['photometry'].keys()}

# Magnitude
magnitude1 = {'Y1': 20, 'Y10': 21}
magnitude2 = {'Y1': 24, 'Y10': 25}
magnitude = numpy.random.uniform(low=magnitude1[tag], high=magnitude2[tag])

filter = (application_dataset['photometry']['mag_i_lsst'] < magnitude)

# Redshift
redshift1 = {'Y1': 0.5, 'Y10': 1.0}
redshift2 = {'Y1': 2.5, 'Y10': 3.0}
redshift = numpy.random.uniform(low=redshift1[tag], high=redshift2[tag])

filter = filter & (application_dataset['photometry']['redshift'] < redshift)

# Color
color1 = {'Y1': 0.5, 'Y10': 0.0}
color2 = {'Y1': 1.5, 'Y10': 1.0}
color = numpy.random.uniform(low=color1[tag], high=color2[tag])

# Angle
angle1 = {'Y1': 0.0, 'Y10': 0.0}
angle2 = {'Y1': numpy.pi / 2, 'Y10': numpy.pi / 3}
angle = numpy.random.uniform(low=angle1[tag], high=angle2[tag])

application_color = application_dataset['photometry']['mag_g_lsst'] - application_dataset['photometry']['mag_z_lsst']
filter = filter & (application_dataset['photometry']['mag_i_lsst'] - magnitude  - numpy.tan(angle) * (application_color - color) < 0)

# Factor
factor1 = {'Y1': 0.5, 'Y10': 1.0}
factor2 = {'Y1': 1.5, 'Y10': 2.0}
factor = numpy.random.uniform(low=factor1[tag], high=factor2[tag])
rate = 1 / (1 + factor * numpy.exp(application_dataset['photometry']['mag_i_lsst'].max() - application_dataset['photometry']['mag_i_lsst']))

# Size
size1 = {'Y1': 100000, 'Y10': 250000}
size2 = {'Y1': 200000, 'Y10': 500000}
size = numpy.minimum(numpy.random.randint(low=size1[tag], high=size2[tag]), numpy.sum(filter))

application_size = len(application_dataset['photometry']['redshift'])
indices = numpy.random.choice(numpy.arange(application_size)[filter], size=size, replace=False, p=rate[filter] / numpy.sum(rate[filter]))

# Degradation
degradation_dataset = {
    'morphology': {key: application_dataset['morphology'][key][indices] for key in application_dataset['morphology'].keys()},
    'photometry': {key: application_dataset['photometry'][key][indices] for key in application_dataset['photometry'].keys()}
}

# SOM
data_store = core.stage.RailStage.data_store
data_store.__class__.allow_overwrite = True

model_name = os.path.join(dataset_folder, '{}/SOM/INFORM.pkl'.format(tag))
model = data_store.read_file(key='model', path=model_name, handle_class=core.data.ModelHandle)()

chunk = 100000
degradation_size = len(degradation_dataset['photometry']['redshift'])
degradation_cell_coordinate = numpy.zeros((degradation_size, 2), dtype=numpy.int32)

for m in range(degradation_size // chunk + 1):
    begin = m * chunk
    stop = min((m + 1) * chunk, degradation_size)
    degradation = {key: degradation_dataset['photometry'][key][begin: stop].astype(numpy.float32) for key in model['usecols']}
    
    degradation_column = somoclu_som._computemagcolordata(data=degradation, mag_column_name=model['ref_column'], column_names=model['usecols'], colusage=model['column_usage'])
    degradation_cell_coordinate[begin: stop, :] = somoclu_som.get_bmus(model['som'], degradation_column)

degradation_cell_coordinate1 = degradation_cell_coordinate[:, 0]
degradation_cell_coordinate2 = degradation_cell_coordinate[:, 1]
degradation_cell_id = numpy.ravel_multi_index(numpy.transpose(degradation_cell_coordinate), (model['n_rows'], model['n_columns']))

cell_size = model['n_rows'] * model['n_columns']
degradation_cell_count = numpy.bincount(degradation_cell_id, minlength=cell_size)
degradation_cell_z_true = numpy.divide(numpy.bincount(degradation_cell_id, weights=degradation_dataset['photometry']['redshift_true'], minlength=cell_size), degradation_cell_count, out=numpy.ones(cell_size) * numpy.nan, where=degradation_cell_count != 0)

# Save
with h5py.File(os.path.join(dataset_folder, '{}/DEGRADATION/DATA{}.hdf5'.format(tag, index)), 'w') as file:
    file.create_group('meta')
    
    file['meta'].create_dataset('size', data=size, dtype=numpy.int32)
    file['meta'].create_dataset('angle', data=angle, dtype=numpy.float32)
    file['meta'].create_dataset('color', data=color, dtype=numpy.float32)
    file['meta'].create_dataset('factor', data=factor, dtype=numpy.float32)
    file['meta'].create_dataset('redshift', data=redshift, dtype=numpy.float32)
    file['meta'].create_dataset('magnitude', data=magnitude, dtype=numpy.float32)
    
    file['meta'].create_dataset('cell_size', data=cell_size, dtype=numpy.int32)
    file['meta'].create_dataset('cell_size1', data=model['n_rows'], dtype=numpy.int32)
    file['meta'].create_dataset('cell_size2', data=model['n_columns'], dtype=numpy.int32)
    
    file['meta'].create_dataset('cell_id', data=degradation_cell_id, dtype=numpy.int32)
    file['meta'].create_dataset('cell_coordinate1', data=degradation_cell_coordinate1, dtype=numpy.int32)
    file['meta'].create_dataset('cell_coordinate2', data=degradation_cell_coordinate2, dtype=numpy.int32)
    
    file['meta'].create_dataset('cell_count', data=degradation_cell_count, dtype=numpy.int32)
    file['meta'].create_dataset('cell_z_true', data=degradation_cell_z_true, dtype=numpy.float32)
    
    file.create_group('morphology')
    for key in degradation_dataset['morphology'].keys():
        file['morphology'].create_dataset(key, data=degradation_dataset['morphology'][key], dtype=degradation_dataset['morphology'][key].dtype)
    
    file.create_group('photometry')
    for key in degradation_dataset['photometry'].keys():
        file['photometry'].create_dataset(key, data=degradation_dataset['photometry'][key], dtype=degradation_dataset['photometry'][key].dtype)

# Duration
end = time.time()
duration = (end - start) / 60
print('Time: {:.2f} minutes'.format(duration))

Selection dataset

In [ ]:
# Start
start = time.time()

# Path
dataset_folder = os.path.join(folder, 'DATASET/')
os.makedirs(os.path.join(dataset_folder, '{}'.format(tag)), exist_ok=True)
os.makedirs(os.path.join(dataset_folder, '{}/SELECTION/'.format(tag)), exist_ok=True)

# Load
with h5py.File(os.path.join(dataset_folder, '{}/SIMULATION/SIMULATION.hdf5'.format(tag)), 'r') as file:
    simulation_dataset = {key: file[key][...] for key in file.keys()}
simulation_size = len(simulation_dataset['redshift'])

with h5py.File(os.path.join(dataset_folder, '{}/APPLICATION/DATA{}.hdf5'.format(tag, index)), 'r') as file:
    application_cell_count = file['meta']['cell_count'][...]
    application_size = len(file['photometry']['redshift'][...])

# SOM
data_store = core.stage.RailStage.data_store
data_store.__class__.allow_overwrite = True

model_name = os.path.join(dataset_folder, '{}/SOM/INFORM.pkl'.format(tag))
model = data_store.read_file(key='model', path=model_name, handle_class=core.data.ModelHandle)()

# Error
error_model = LsstErrorModel(
    sigLim=1.0,
    absFlux=True,
    ndMode='sigLim', 
    majorCol='major', 
    minorCol='minor', 
    decorrelate=True,
    extendedType='auto', 
    nYrObs=int(tag[1:]), 
    renameDict={
        'u': 'mag_u_lsst',
        'g': 'mag_g_lsst',
        'r': 'mag_r_lsst',
        'i': 'mag_i_lsst',
        'z': 'mag_z_lsst',
        'y': 'mag_y_lsst'
    }
)

simulation_dataset = dict(error_model(pandas.DataFrame(simulation_dataset), random_state=index))

chunk = 100000
simulation_cell_coordinate = numpy.zeros((simulation_size, 2), dtype=numpy.int32)

for m in range(simulation_size // chunk + 1):
    begin = m * chunk
    stop = min((m + 1) * chunk, simulation_size)
    simulation = {key: simulation_dataset[key][begin: stop].astype(numpy.float32) for key in model['usecols']}
    
    simulation_column = somoclu_som._computemagcolordata(data=simulation, mag_column_name=model['ref_column'], column_names=model['usecols'], colusage='colors')
    simulation_cell_coordinate[begin: stop, :] = somoclu_som.get_bmus(model['som'], simulation_column)

simulation_cell_coordinate1 = simulation_cell_coordinate[:, 0]
simulation_cell_coordinate2 = simulation_cell_coordinate[:, 1]
simulation_cell_id = numpy.ravel_multi_index(numpy.transpose(simulation_cell_coordinate), (model['n_rows'], model['n_columns']))

cell_size = model['n_rows'] * model['n_columns']
simulation_cell_count = numpy.bincount(simulation_cell_id, minlength=cell_size)

simulation_weight = numpy.divide(application_cell_count, simulation_cell_count, out=numpy.zeros(cell_size), where=simulation_cell_count != 0)
simulation_probability = simulation_weight[simulation_cell_id] / numpy.sum(simulation_weight[simulation_cell_id])

# Selection
indices = numpy.random.choice(simulation_size, size=application_size, replace=False, p=simulation_probability)
selection_dataset = {key: simulation_dataset[key][indices].astype(numpy.float32) for key in simulation_dataset.keys()}

selection_cell_coordinate1 = simulation_cell_coordinate1[indices]
selection_cell_coordinate2 = simulation_cell_coordinate2[indices]
selection_cell_id = simulation_cell_id[indices]

selection_cell_count = numpy.bincount(selection_cell_id, minlength=cell_size)
selection_cell_z_true = numpy.divide(numpy.bincount(selection_cell_id, weights=selection_dataset['redshift_true'], minlength=cell_size), selection_cell_count, out=numpy.ones(cell_size) * numpy.nan, where=selection_cell_count != 0)

# Save
with h5py.File(os.path.join(dataset_folder, '{}/SELECTION/DATA{}.hdf5'.format(tag, index)), 'w') as file:
    file.create_group('meta')
    
    file['meta'].create_dataset('cell_size', data=cell_size, dtype=numpy.int32)
    file['meta'].create_dataset('cell_size1', data=model['n_rows'], dtype=numpy.int32)
    file['meta'].create_dataset('cell_size2', data=model['n_columns'], dtype=numpy.int32)
    
    file['meta'].create_dataset('cell_id', data=selection_cell_id, dtype=numpy.int32)
    file['meta'].create_dataset('cell_coordinate1', data=selection_cell_coordinate1, dtype=numpy.int32)
    file['meta'].create_dataset('cell_coordinate2', data=selection_cell_coordinate2, dtype=numpy.int32)
    
    file['meta'].create_dataset('cell_count', data=selection_cell_count, dtype=numpy.int32)
    file['meta'].create_dataset('cell_z_true', data=selection_cell_z_true, dtype=numpy.float32)
    
    file.create_group('photometry')
    file['photometry'].create_dataset('redshift', data=selection_dataset['redshift'], dtype=numpy.float32)
    file['photometry'].create_dataset('redshift_true', data=selection_dataset['redshift_true'], dtype=numpy.float32)
    
    file['photometry'].create_dataset('mag_u_lsst', data=selection_dataset['mag_u_lsst'], dtype=numpy.float32)
    file['photometry'].create_dataset('mag_g_lsst', data=selection_dataset['mag_g_lsst'], dtype=numpy.float32)
    file['photometry'].create_dataset('mag_r_lsst', data=selection_dataset['mag_r_lsst'], dtype=numpy.float32)
    file['photometry'].create_dataset('mag_i_lsst', data=selection_dataset['mag_i_lsst'], dtype=numpy.float32)
    file['photometry'].create_dataset('mag_z_lsst', data=selection_dataset['mag_z_lsst'], dtype=numpy.float32)
    file['photometry'].create_dataset('mag_y_lsst', data=selection_dataset['mag_y_lsst'], dtype=numpy.float32)
    
    file['photometry'].create_dataset('mag_u_lsst_err', data=selection_dataset['mag_u_lsst_err'], dtype=numpy.float32)
    file['photometry'].create_dataset('mag_g_lsst_err', data=selection_dataset['mag_g_lsst_err'], dtype=numpy.float32)
    file['photometry'].create_dataset('mag_r_lsst_err', data=selection_dataset['mag_r_lsst_err'], dtype=numpy.float32)
    file['photometry'].create_dataset('mag_i_lsst_err', data=selection_dataset['mag_i_lsst_err'], dtype=numpy.float32)
    file['photometry'].create_dataset('mag_z_lsst_err', data=selection_dataset['mag_z_lsst_err'], dtype=numpy.float32)
    file['photometry'].create_dataset('mag_y_lsst_err', data=selection_dataset['mag_y_lsst_err'], dtype=numpy.float32)
    
    file.create_group('morphology')
    file['morphology'].create_dataset('ra', data=selection_dataset['ra'], dtype=numpy.float32)
    file['morphology'].create_dataset('dec', data=selection_dataset['dec'], dtype=numpy.float32)
    
    file['morphology'].create_dataset('id', data=selection_dataset['id'], dtype=numpy.int32)
    file['morphology'].create_dataset('value', data=selection_dataset['value'], dtype=numpy.int32)
    
    file['morphology'].create_dataset('mu', data=selection_dataset['mu'], dtype=numpy.float32)
    file['morphology'].create_dataset('eta', data=selection_dataset['eta'], dtype=numpy.float32)
    file['morphology'].create_dataset('sigma', data=selection_dataset['sigma'], dtype=numpy.float32)
    
    file['morphology'].create_dataset('major', data=selection_dataset['major'], dtype=numpy.float32)
    file['morphology'].create_dataset('minor', data=selection_dataset['minor'], dtype=numpy.float32)
    
    file['morphology'].create_dataset('major_disk', data=selection_dataset['major_disk'], dtype=numpy.float32)
    file['morphology'].create_dataset('major_bulge', data=selection_dataset['major_bulge'], dtype=numpy.float32)
    
    file['morphology'].create_dataset('ellipticity_disk', data=selection_dataset['ellipticity_disk'], dtype=numpy.float32)
    file['morphology'].create_dataset('ellipticity_bulge', data=selection_dataset['ellipticity_bulge'], dtype=numpy.float32)
    file['morphology'].create_dataset('bulge_to_total_ratio', data=selection_dataset['bulge_to_total_ratio'], dtype=numpy.float32)

# Duration
end = time.time()
duration = (end - start) / 60
print('Time: {:.2f} minutes'.format(duration))

Augmentation dataset

In [ ]:
# Start
start = time.time()

# Path
dataset_folder = os.path.join(folder, 'DATASET/')
os.makedirs(os.path.join(dataset_folder, '{}'.format(tag)), exist_ok=True)
os.makedirs(os.path.join(dataset_folder, '{}/AUGMENTATION/'.format(tag)), exist_ok=True)

# Degradation
with h5py.File(os.path.join(dataset_folder, '{}/DEGRADATION/DATA{}.hdf5'.format(tag, index)), 'r') as file:
    degradation_cell_size = file['meta']['cell_size'][...]
    degradation_cell_count = file['meta']['cell_count'][...]
    degradation_redshift = file['photometry']['redshift'][...]
    degradation_magnitude = file['photometry']['mag_i_lsst'][...]
degradation_size = len(degradation_redshift)

# Selection
selection_dataset = {
    'meta': {},
    'morphology': {},
    'photometry': {}
}

with h5py.File(os.path.join(dataset_folder, '{}/SELECTION/DATA{}.hdf5'.format(tag, index)), 'r') as file:
    selection_dataset['meta'] = {key: file['meta'][key][...] for key in file['meta'].keys()}
    selection_dataset['morphology'] = {key: file['morphology'][key][...] for key in file['morphology'].keys()}
    selection_dataset['photometry'] = {key: file['photometry'][key][...] for key in file['photometry'].keys()}

selection_cell_id = selection_dataset['meta']['cell_id']
selection_size = len(selection_dataset['photometry']['redshift'])

fraction = numpy.sum(degradation_cell_count == 0) / degradation_cell_size
filter = numpy.isin(selection_cell_id, numpy.arange(degradation_cell_size)[degradation_cell_count == 0])

# Magnitude
magnitude = numpy.max(degradation_magnitude)
filter = filter | (selection_dataset['photometry']['mag_i_lsst'] > magnitude)

# Redshift
redshift = numpy.max(degradation_redshift)
filter = filter | (selection_dataset['photometry']['redshift'] > redshift)

# Fraction
size1 = int(0.1 * degradation_size)
size2 = int(1.0 * degradation_size)
size = numpy.random.randint(low=size1, high=size2)
indices = numpy.random.choice(numpy.arange(selection_size)[filter], size=size, replace=False)

# Augmentation
augmentation_dataset = {
    'morphology': {key: selection_dataset['morphology'][key][indices] for key in selection_dataset['morphology'].keys()},
    'photometry': {key: selection_dataset['photometry'][key][indices] for key in selection_dataset['photometry'].keys()}
}

# SOM
data_store = core.stage.RailStage.data_store
data_store.__class__.allow_overwrite = True

model_name = os.path.join(dataset_folder, '{}/SOM/INFORM.pkl'.format(tag))
model = data_store.read_file(key='model', path=model_name, handle_class=core.data.ModelHandle)()

chunk = 100000
augmentation_size = len(augmentation_dataset['photometry']['redshift'])
augmentation_cell_coordinate = numpy.zeros((augmentation_size, 2), dtype=numpy.int32)

for m in range(augmentation_size // chunk + 1):
    begin = m * chunk
    stop = min((m + 1) * chunk, augmentation_size)
    augmentation = {key: augmentation_dataset['photometry'][key][begin: stop].astype(numpy.float32) for key in model['usecols']}
    
    augmentation_column = somoclu_som._computemagcolordata(data=augmentation, mag_column_name=model['ref_column'], column_names=model['usecols'], colusage=model['column_usage'])
    augmentation_cell_coordinate[begin: stop, :] = somoclu_som.get_bmus(model['som'], augmentation_column)

augmentation_cell_coordinate1 = augmentation_cell_coordinate[:, 0]
augmentation_cell_coordinate2 = augmentation_cell_coordinate[:, 1]
augmentation_cell_id = numpy.ravel_multi_index(numpy.transpose(augmentation_cell_coordinate), dims=(model['n_rows'], model['n_columns']))

cell_size = model['n_rows'] * model['n_columns']
augmentation_cell_count = numpy.bincount(augmentation_cell_id, minlength=cell_size)
augmentation_cell_z_true = numpy.divide(numpy.bincount(augmentation_cell_id, weights=augmentation_dataset['photometry']['redshift_true'], minlength=cell_size), augmentation_cell_count, out=numpy.ones(cell_size) * numpy.nan, where=augmentation_cell_count != 0)

# Save
with h5py.File(os.path.join(dataset_folder, '{}/AUGMENTATION/DATA{}.hdf5'.format(tag, index)), 'w') as file:
    file.create_group('meta')
    
    file['meta'].create_dataset('size', data=size, dtype=numpy.int32)
    file['meta'].create_dataset('fraction', data=fraction, dtype=numpy.float32)
    file['meta'].create_dataset('redshift', data=redshift, dtype=numpy.float32)
    file['meta'].create_dataset('magnitude', data=magnitude, dtype=numpy.float32)
    
    file['meta'].create_dataset('cell_size', data=cell_size, dtype=numpy.int32)
    file['meta'].create_dataset('cell_size1', data=model['n_rows'], dtype=numpy.int32)
    file['meta'].create_dataset('cell_size2', data=model['n_columns'], dtype=numpy.int32)
    
    file['meta'].create_dataset('cell_id', data=augmentation_cell_id, dtype=numpy.int32)
    file['meta'].create_dataset('cell_coordinate1', data=augmentation_cell_coordinate1, dtype=numpy.int32)
    file['meta'].create_dataset('cell_coordinate2', data=augmentation_cell_coordinate2, dtype=numpy.int32)
    
    file['meta'].create_dataset('cell_count', data=augmentation_cell_count, dtype=numpy.int32)
    file['meta'].create_dataset('cell_z_true', data=augmentation_cell_z_true, dtype=numpy.float32)
    
    file.create_group('morphology')
    for key in selection_dataset['morphology'].keys():
        file['morphology'].create_dataset(key, data=augmentation_dataset['morphology'][key], dtype=augmentation_dataset['morphology'][key].dtype)
    
    file.create_group('photometry')
    for key in selection_dataset['photometry'].keys():
        file['photometry'].create_dataset(key, data=augmentation_dataset['photometry'][key], dtype=augmentation_dataset['photometry'][key].dtype)

# Duration
end = time.time()
duration = (end - start) / 60
print('Time: {:.2f} minutes'.format(duration))


Combination datasets

In [ ]:
# Start
start = time.time()

# Path
dataset_folder = os.path.join(folder, 'DATASET/')

# Degradation
degradation_dataset = {
    'meta': {},
    'morphology': {},
    'photometry': {}
}

with h5py.File(os.path.join(dataset_folder, '{}/DEGRADATION/DATA{}.hdf5'.format(tag, index)), 'r') as file:
    
    degradation_dataset['meta']['size'] = file['meta']['size'][...]
    degradation_dataset['meta']['angle'] = file['meta']['angle'][...]
    degradation_dataset['meta']['color'] = file['meta']['color'][...]
    degradation_dataset['meta']['factor'] = file['meta']['factor'][...]
    degradation_dataset['meta']['redshift'] = file['meta']['redshift'][...]
    degradation_dataset['meta']['magnitude'] = file['meta']['magnitude'][...]
    
    degradation_dataset['meta']['cell_size'] = file['meta']['cell_size'][...]
    degradation_dataset['meta']['cell_size1'] = file['meta']['cell_size1'][...]
    degradation_dataset['meta']['cell_size2'] = file['meta']['cell_size2'][...]
    
    degradation_dataset['meta']['cell_id'] = file['meta']['cell_id'][...]
    degradation_dataset['meta']['cell_coordinate1'] = file['meta']['cell_coordinate1'][...]
    degradation_dataset['meta']['cell_coordinate2'] = file['meta']['cell_coordinate2'][...]
    
    degradation_dataset['meta']['cell_count'] = file['meta']['cell_count'][...]
    degradation_dataset['meta']['cell_z_true'] = numpy.nan_to_num(file['meta']['cell_z_true'][...])
    
    degradation_dataset['morphology'] = {key: file['morphology'][key][...] for key in file['morphology'].keys()}
    degradation_dataset['photometry'] = {key: file['photometry'][key][...] for key in file['photometry'].keys()}

# Augmentation
augmentation_dataset = {
    'meta': {},
    'morphology': {},
    'photometry': {}
}

with h5py.File(os.path.join(dataset_folder, '{}/AUGMENTATION/DATA{}.hdf5'.format(tag, index)), 'r') as file:
    augmentation_dataset['meta']['size'] = file['meta']['size'][...]
    augmentation_dataset['meta']['fraction'] = file['meta']['fraction'][...]
    augmentation_dataset['meta']['redshift'] = file['meta']['redshift'][...]
    augmentation_dataset['meta']['magnitude'] = file['meta']['magnitude'][...]
    
    augmentation_dataset['meta']['cell_size'] = file['meta']['cell_size'][...]
    augmentation_dataset['meta']['cell_size1'] = file['meta']['cell_size1'][...]
    augmentation_dataset['meta']['cell_size2'] = file['meta']['cell_size2'][...]
    
    augmentation_dataset['meta']['cell_id'] = file['meta']['cell_id'][...]
    augmentation_dataset['meta']['cell_coordinate1'] = file['meta']['cell_coordinate1'][...]
    augmentation_dataset['meta']['cell_coordinate2'] = file['meta']['cell_coordinate2'][...]
    
    augmentation_dataset['meta']['cell_count'] = file['meta']['cell_count'][...]
    augmentation_dataset['meta']['cell_z_true'] = numpy.nan_to_num(file['meta']['cell_z_true'][...])
    
    augmentation_dataset['morphology'] = {key: file['morphology'][key][...] for key in file['morphology'].keys()}
    augmentation_dataset['photometry'] = {key: file['photometry'][key][...] for key in file['photometry'].keys()}

# Combine
cell_size = numpy.average([degradation_dataset['meta']['cell_size'], augmentation_dataset['meta']['cell_size']], axis=0)
cell_size1 = numpy.average([degradation_dataset['meta']['cell_size1'], augmentation_dataset['meta']['cell_size1']], axis=0)
cell_size2 = numpy.average([degradation_dataset['meta']['cell_size2'], augmentation_dataset['meta']['cell_size2']], axis=0)

cell_id = numpy.append(degradation_dataset['meta']['cell_id'], augmentation_dataset['meta']['cell_id'], axis=0)
cell_coordinate1 = numpy.append(degradation_dataset['meta']['cell_coordinate1'], augmentation_dataset['meta']['cell_coordinate1'], axis=0)
cell_coordinate2 = numpy.append(degradation_dataset['meta']['cell_coordinate2'], augmentation_dataset['meta']['cell_coordinate2'], axis=0)

degradation_summation = degradation_dataset['meta']['cell_z_true'] * degradation_dataset['meta']['cell_count']
augmentation_summation = augmentation_dataset['meta']['cell_z_true'] * augmentation_dataset['meta']['cell_count']

cell_count = degradation_dataset['meta']['cell_count'] + augmentation_dataset['meta']['cell_count']
cell_z_true = numpy.divide(degradation_summation + augmentation_summation, cell_count, out=numpy.ones_like(cell_count) * numpy.nan, where=cell_count != 0)

# Save
os.makedirs(os.path.join(dataset_folder, '{}/'.format(tag)), exist_ok=True)
os.makedirs(os.path.join(dataset_folder, '{}/COMBINATION/'.format(tag)), exist_ok=True)

with h5py.File(os.path.join(dataset_folder, '{}/COMBINATION/DATA{}.hdf5'.format(tag, index)), 'w') as file:
    file.create_group('meta')
    
    file['meta'].create_dataset('degradation_size', data=degradation_dataset['meta']['size'], dtype=numpy.int32)
    file['meta'].create_dataset('degradation_angle', data=degradation_dataset['meta']['angle'], dtype=numpy.float32)
    file['meta'].create_dataset('degradation_color', data=degradation_dataset['meta']['color'], dtype=numpy.float32)
    file['meta'].create_dataset('degradation_factor', data=degradation_dataset['meta']['factor'], dtype=numpy.float32)
    file['meta'].create_dataset('degradation_redshift', data=degradation_dataset['meta']['redshift'], dtype=numpy.float32)
    file['meta'].create_dataset('degradation_magnitude', data=degradation_dataset['meta']['magnitude'], dtype=numpy.float32)
    
    file['meta'].create_dataset('augmentation_size', data=augmentation_dataset['meta']['size'], dtype=numpy.int32)
    file['meta'].create_dataset('augmentation_fraction', data=augmentation_dataset['meta']['fraction'], dtype=numpy.float32)
    file['meta'].create_dataset('augmentation_redshift', data=augmentation_dataset['meta']['redshift'], dtype=numpy.float32)
    file['meta'].create_dataset('augmentation_magnitude', data=augmentation_dataset['meta']['magnitude'], dtype=numpy.float32)
    
    file['meta'].create_dataset('cell_size', data=cell_size, dtype=numpy.int32)
    file['meta'].create_dataset('cell_size1', data=cell_size1, dtype=numpy.int32)
    file['meta'].create_dataset('cell_size2', data=cell_size2, dtype=numpy.int32)
    
    file['meta'].create_dataset('cell_id', data=cell_id, dtype=numpy.int32)
    file['meta'].create_dataset('cell_coordinate1', data=cell_coordinate1, dtype=numpy.float32)
    file['meta'].create_dataset('cell_coordinate2', data=cell_coordinate2, dtype=numpy.float32)
    
    file['meta'].create_dataset('cell_count', data=cell_count, dtype=numpy.int32)
    file['meta'].create_dataset('cell_z_true', data=cell_z_true, dtype=numpy.float32)
    
    file.create_group('morphology')
    for key in degradation_dataset['morphology'].keys():
        value = numpy.append(degradation_dataset['morphology'][key], augmentation_dataset['morphology'][key], axis=0)
        file['morphology'].create_dataset(key, data=value, dtype=value.dtype)
    
    file.create_group('photometry')
    for key in degradation_dataset['photometry'].keys():
        value = numpy.append(degradation_dataset['photometry'][key], augmentation_dataset['photometry'][key], axis=0)
        file['photometry'].create_dataset(key, data=value, dtype=value.dtype)

# Duration
end = time.time()
duration = (end - start) / 60
print('Time: {:.2f} minutes'.format(duration))

Redshift distribution

In [ ]:
# Start
start = time.time()

# Path
figure_folder = os.path.join(folder, 'FIGURE/')
dataset_folder = os.path.join(folder, 'DATASET/')

# Plot
os.environ['PATH'] = '/global/homes/y/yhzhang/opt/texlive/bin/x86_64-linux:' + os.environ['PATH']
pyplot.rcParams['text.latex.preamble'] = r'\usepackage{amsmath}'
pyplot.rcParams['pgf.texsystem'] = 'pdflatex'
pyplot.rcParams['text.usetex'] = True
pyplot.rcParams['font.size'] = 20

# Application
with h5py.File(os.path.join(dataset_folder, '{}/APPLICATION/DATA{}.hdf5'.format(tag, index)), 'r') as file:
    application_redshift_true = file['photometry']['redshift_true'][...]

# Degradation
with h5py.File(os.path.join(dataset_folder, '{}/DEGRADATION/DATA{}.hdf5'.format(tag, index)), 'r') as file:
    degradation_redshift_true = file['photometry']['redshift_true'][...]

# Augmentation
with h5py.File(os.path.join(dataset_folder, '{}/AUGMENTATION/DATA{}.hdf5'.format(tag, index)), 'r') as file:
    augmentation_redshift_true = file['photometry']['redshift_true'][...]

# Combination
with h5py.File(os.path.join(dataset_folder, '{}/COMBINATION/DATA{}.hdf5'.format(tag, index)), 'r') as file:
    combination_redshift_true = file['photometry']['redshift_true'][...]

# Bin
z1 = 0.0
z2 = 3.0
z_size = 100
z_bin = numpy.linspace(z1, z2, z_size + 1)

figure, plot = pyplot.subplots(nrows=1, ncols=1, figsize=(12, 6))

plot.hist(application_redshift_true, bins=z_bin, linewidth=4.0, density=True, histtype='step', color='black', label=r'$\mathrm{application}$')

plot.hist(degradation_redshift_true, bins=z_bin, linewidth=4.0, density=True, histtype='step', color='darkred', label=r'$\mathrm{degradation}$')

plot.hist(augmentation_redshift_true, bins=z_bin, linewidth=4.0, density=True, histtype='step', color='darkorange', label=r'$\mathrm{augmentation}$')

plot.hist(combination_redshift_true, bins=z_bin, linewidth=4.0, density=True, histtype='step', color='darkblue', label=r'$\mathrm{combination}$')

plot.legend()
plot.set_xlim(z_bin.min(), z_bin.max())

plot.set_xlabel(r'$z_\mathrm{true}$')
plot.set_ylabel(r'$\mathcal{P}(z)$')

os.makedirs(os.path.join(figure_folder, '{}/'.format(tag)), exist_ok=True)
os.makedirs(os.path.join(figure_folder, '{}/DISTRIBUTION/'.format(tag)), exist_ok=True)
figure.savefig(os.path.join(figure_folder, '{}/DISTRIBUTION/FIGURE{}.png'.format(tag, index)), format='png', bbox_inches='tight', dpi=512)

# Duration
end = time.time()
duration = (end - start) / 60
print('Time: {:.2f} minutes'.format(duration))

Color - magnitude diagram

In [ ]:
# Start
start = time.time()

# Path
figure_folder = os.path.join(folder, 'FIGURE/')
dataset_folder = os.path.join(folder, 'DATASET/')

# Plot
os.environ['PATH'] = '/global/homes/y/yhzhang/opt/texlive/bin/x86_64-linux:' + os.environ['PATH']
pyplot.rcParams['text.latex.preamble'] = r'\usepackage{amsmath}'
pyplot.rcParams['pgf.texsystem'] = 'pdflatex'
pyplot.rcParams['text.usetex'] = True
pyplot.rcParams['font.size'] = 20

# Application
with h5py.File(os.path.join(dataset_folder, '{}/APPLICATION/DATA{}.hdf5'.format(tag, index)), 'r') as file:
    application_magnitude = file['photometry']['mag_i_lsst'][...]
    application_color = file['photometry']['mag_g_lsst'][...] - file['photometry']['mag_z_lsst'][...]

# Degradation
with h5py.File(os.path.join(dataset_folder, '{}/DEGRADATION/DATA{}.hdf5'.format(tag, index)), 'r') as file:
    degradation_magnitude = file['photometry']['mag_i_lsst'][...]
    degradation_color = file['photometry']['mag_g_lsst'][...] - file['photometry']['mag_z_lsst'][...]

# Augmentation
with h5py.File(os.path.join(dataset_folder, '{}/AUGMENTATION/DATA{}.hdf5'.format(tag, index)), 'r') as file:
    augmentation_magnitude = file['photometry']['mag_i_lsst'][...]
    augmentation_color = file['photometry']['mag_g_lsst'][...] - file['photometry']['mag_z_lsst'][...]

# Combination
with h5py.File(os.path.join(dataset_folder, '{}/COMBINATION/DATA{}.hdf5'.format(tag, index)), 'r') as file:
    combination_magnitude = file['photometry']['mag_i_lsst'][...]
    combination_color = file['photometry']['mag_g_lsst'][...] - file['photometry']['mag_z_lsst'][...]

# Bin
magnitude1 = 16.0
magnitude2 = 26.0
magnitude_size = 100
magnitude_bin = numpy.linspace(magnitude1, magnitude2, magnitude_size + 1)

color1 = -2.0
color2 = +6.0
color_size = 100
color_bin = numpy.linspace(color1, color2, color_size + 1)

figure = pyplot.figure(figsize = (12, 16))
normalize = colors.LogNorm(vmin=1, vmax=10000)
gridspec = GridSpec(nrows=2, ncols=2, figure=figure, top=0.70, bottom=0.15, hspace=0.0, wspace=0.0)

plot = figure.add_subplot(gridspec[0, 0])

plot.text(3.0, 17.0, r'$\mathrm{application}$')

image = plot.hist2d(application_color, application_magnitude, bins=[color_bin, magnitude_bin], norm=normalize, cmap='magma')[-1]

plot.set_ylabel(r'$i$')
plot.set_xticklabels([])

plot.get_yticklabels()[0].set_visible(False)
plot.get_xticklabels()[0].set_visible(False)

plot.set_xlim(color_bin.min(), color_bin.max())
plot.set_ylim(magnitude_bin.min(), magnitude_bin.max())

plot = figure.add_subplot(gridspec[0, 1])

plot.text(3.0, 17.0, r'$\mathrm{degradation}$')

image = plot.hist2d(degradation_color, degradation_magnitude, bins=[color_bin, magnitude_bin], norm=normalize, cmap='magma')[-1]

plot.set_yticklabels([])
plot.set_xticklabels([])

plot.get_yticklabels()[0].set_visible(False)
plot.get_xticklabels()[0].set_visible(False)

plot.set_xlim(color_bin.min(), color_bin.max())
plot.set_ylim(magnitude_bin.min(), magnitude_bin.max())

plot = figure.add_subplot(gridspec[1, 0])

plot.text(3.0, 17.0, r'$\mathrm{augmentation}$')

image = plot.hist2d(augmentation_color, augmentation_magnitude, bins=[color_bin, magnitude_bin], norm=normalize, cmap='magma')[-1]

plot.set_xlim(color_bin.min(), color_bin.max())
plot.set_ylim(magnitude_bin.min(), magnitude_bin.max())

plot.get_yticklabels()[0].set_visible(False)
plot.get_xticklabels()[0].set_visible(False)

plot.set_ylabel(r'$i$')
plot.set_xlabel(r'$g - z$')

plot = figure.add_subplot(gridspec[1, 1])

plot.text(3.0, 17.0, r'$\mathrm{combination}$')

image = plot.hist2d(combination_color, combination_magnitude, bins=[color_bin, magnitude_bin], norm=normalize, cmap='magma')[-1]

plot.set_xlim(color_bin.min(), color_bin.max())
plot.set_ylim(magnitude_bin.min(), magnitude_bin.max())

plot.get_yticklabels()[0].set_visible(False)
plot.get_xticklabels()[0].set_visible(False)

plot.set_yticklabels([])
plot.set_xlabel(r'$g - z$')

color_bar = figure.colorbar(image, cax=figure.add_axes([0.15, 0.05, 0.70, 0.05]), orientation='horizontal')
color_bar.set_label(r'$\mathrm{Counts}$')
figure.subplots_adjust(bottom=0.15)

os.makedirs(os.path.join(figure_folder, '{}/'.format(tag)), exist_ok=True)
os.makedirs(os.path.join(figure_folder, '{}/DIAGRAM/'.format(tag)), exist_ok=True)

figure.savefig(os.path.join(figure_folder, '{}/DIAGRAM/FIGURE{}.png'.format(tag, index)), format='png', bbox_inches='tight', dpi=512)
pyplot.close(figure)

# Duration
end = time.time()
duration = (end - start) / 60
print('Time: {:.2f} minutes'.format(duration))

SOM maps

In [ ]:

# Start
start = time.time()
print('Index: {}'.format(index))

# Path
figure_folder = os.path.join(folder, 'FIGURE/')
dataset_folder = os.path.join(folder, 'DATASET/')

# Plot
os.environ['PATH'] = '/global/homes/y/yhzhang/opt/texlive/bin/x86_64-linux:' + os.environ['PATH']
pyplot.rcParams['text.latex.preamble'] = r'\usepackage{amsmath}'
pyplot.rcParams['pgf.texsystem'] = 'pdflatex'
pyplot.rcParams['text.usetex'] = True
pyplot.rcParams['font.size'] = 20

# Application
with h5py.File(os.path.join(dataset_folder, '{}/APPLICATION/DATA{}.hdf5'.format(tag, index)), 'r') as file:
    cell_size1 = file['meta']['cell_size1'][...]
    cell_size2 = file['meta']['cell_size2'][...]
    application_cell_z_true = file['meta']['cell_z_true'][...]
application_map = application_cell_z_true.reshape((cell_size1, cell_size2))

# Degradation
with h5py.File(os.path.join(dataset_folder, '{}/DEGRADATION/DATA{}.hdf5'.format(tag, index)), 'r') as file:
    cell_size1 = file['meta']['cell_size1'][...]
    cell_size2 = file['meta']['cell_size2'][...]
    degradation_cell_z_true = file['meta']['cell_z_true'][...]
degradation_map = degradation_cell_z_true.reshape((cell_size1, cell_size2))

# Augmentation
with h5py.File(os.path.join(dataset_folder, '{}/AUGMENTATION/DATA{}.hdf5'.format(tag, index)), 'r') as file:
    cell_size1 = file['meta']['cell_size1'][...]
    cell_size2 = file['meta']['cell_size2'][...]
    augmentation_cell_z_true = file['meta']['cell_z_true'][...]
augmentation_map = augmentation_cell_z_true.reshape((cell_size1, cell_size2))

# Combination
with h5py.File(os.path.join(dataset_folder, '{}/COMBINATION/DATA{}.hdf5'.format(tag, index)), 'r') as file:
    cell_size1 = file['meta']['cell_size1'][...]
    cell_size2 = file['meta']['cell_size2'][...]
    combination_cell_z_true = file['meta']['cell_z_true'][...]
combination_map = combination_cell_z_true.reshape((cell_size1, cell_size2))

# Plot
normalize = colors.Normalize(vmin=0.0, vmax=2.0)
figure, plot = pyplot.subplots(nrows=2, ncols=2, figsize=(12, 12))

mesh = plot[0, 0].imshow(application_map, norm=normalize, cmap='coolwarm')
plot[0, 0].set_title(r'$\mathrm{application}$')
plot[0, 0].axis('off')

mesh = plot[0, 1].imshow(degradation_map, norm=normalize, cmap='coolwarm')
plot[0, 1].set_title(r'$\mathrm{degradation}$')
plot[0, 1].axis('off')

mesh = plot[1, 0].imshow(augmentation_map, norm=normalize, cmap='coolwarm')
plot[1, 0].set_title(r'$\mathrm{augmentation}$')
plot[1, 0].axis('off')

mesh = plot[1, 1].imshow(combination_map, norm=normalize, cmap='coolwarm')
plot[1, 1].set_title(r'$\mathrm{combination}$')
plot[1, 1].axis('off')

color_bar = figure.colorbar(mesh, cax=figure.add_axes([0.15, 0.05, 0.70, 0.05]), orientation='horizontal')
figure.subplots_adjust(bottom=0.12, hspace=0.10, wspace=0.05)
color_bar.set_label(r'$\langle z_\mathrm{true} \rangle$')

os.makedirs(os.path.join(figure_folder, '{}/'.format(tag)), exist_ok=True)
os.makedirs(os.path.join(figure_folder, '{}/MAP/'.format(tag)), exist_ok=True)

figure.savefig(os.path.join(figure_folder, '{}/MAP/FIGURE{}.png'.format(tag, index)), format='png', bbox_inches='tight', dpi=512)
pyplot.close(figure)

# Duration
end = time.time()
duration = (end - start) / 60
print('Time: {:.2f} minutes'.format(duration))